# Unit 2 — Advanced Visualization & Storytelling (UE24CS342AA9)
## Activity Notebook: Building the Global Development Monitor

### Scenario

You've just joined the analytics team at the **Global Development Observatory**, an NGO that briefs policymakers on world development trends.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
import shap
import ipywidgets as widgets
from ipywidgets import interact, Dropdown
np.random.seed(0)
gapminder = px.data.gapminder()
centroids = pd.read_csv("country_centroids_computed.csv")
gap_latest = gapminder[gapminder.year == 2007].merge(centroids, on="iso_alpha", how="left")
gap_latest = gap_latest.assign(total_gdp=gap_latest["pop"] * gap_latest.gdpPercap)
print(f"Loaded {gapminder.country.nunique()} countries, years {sorted(gapminder.year.unique())}.")


In [ ]:
fig1 = px.choropleth(gap_latest, locations="iso_alpha", color="total_gdp", hover_name="country", color_continuous_scale="Blues", title="COUNT view: Total GDP by country (2007)")
fig1.show()
fig2 = px.choropleth(gap_latest, locations="iso_alpha", color="gdpPercap", hover_name="country", color_continuous_scale="Greens", title="RATE view: GDP per Capita by country (2007)")
fig2.show()
top_total = gap_latest.nlargest(5, "total_gdp")[["country", "total_gdp"]]
top_rate = gap_latest.nlargest(5, "gdpPercap")[["country", "gdpPercap"]]
print("Top 5 by total GDP:\n", top_total)
print("\nTop 5 by GDP per capita:\n", top_rate)


In [ ]:
X_model = gapminder[["gdpPercap", "pop", "year"]].values
y_model = gapminder["lifeExp"].values
rf = RandomForestRegressor(n_estimators=200, random_state=0)
rf.fit(X_model, y_model)
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_model)
row_idx = gapminder.index[(gapminder.country == "Norway") & (gapminder.year == 2007)][0]
row_pos = gapminder.index.get_loc(row_idx)
vals = shap_values[row_pos]
feature_names = ["gdpPercap", "pop", "year"]
colors = ["#3b6ea5" if v > 0 else "#b5432e" for v in vals]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(feature_names, vals, color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("SHAP value")
ax.set_title("Local SHAP explanation: Norway, 2007")
plt.show()


In [ ]:
def bootstrap_ci(values, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    boot_means = np.array([rng.choice(values, size=len(values), replace=True).mean() for _ in range(n_boot)])
    return values.mean(), np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5)
continent_a, continent_b = "Africa", "Europe"
values_a = gap_latest[gap_latest.continent == continent_a].lifeExp.values
values_b = gap_latest[gap_latest.continent == continent_b].lifeExp.values
result_a = bootstrap_ci(values_a); result_b = bootstrap_ci(values_b)
fig, ax = plt.subplots(figsize=(5, 4))
for i, (name, res) in enumerate([(continent_a, result_a), (continent_b, result_b)]):
    mean, lo, hi = res
    ax.errorbar(i, mean, yerr=[[mean - lo], [hi - mean]], fmt="o", capsize=6, markersize=8)
ax.set_xticks([0, 1]); ax.set_xticklabels([continent_a, continent_b])
ax.set_ylabel("Mean Life Expectancy (2007), 95% CI")
ax.set_title("Bootstrap Confidence Intervals")
plt.tight_layout(); plt.show()
print(continent_a, result_a); print(continent_b, result_b)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
sc = axes[0].scatter(gap_latest.centroid_lon, gap_latest.centroid_lat, c=gap_latest.lifeExp, cmap="viridis", s=25)
fig.colorbar(sc, ax=axes[0], label="Life Expectancy")
axes[0].set_title("Map: Country Centroids by Life Expectancy")
axes[0].set_xlabel("Longitude"); axes[0].set_ylabel("Latitude")
world_trend = gapminder.groupby("year").lifeExp.mean()
axes[1].plot(world_trend.index, world_trend.values, marker="o", color="navy")
axes[1].set_title("World Avg Life Expectancy by Year")
axes[1].set_xlabel("Year"); axes[1].set_ylabel("Life Expectancy")
pop_by_cont = gap_latest.groupby("continent")["pop"].sum().sort_values()
axes[2].barh(pop_by_cont.index, pop_by_cont.values, color="teal")
axes[2].set_title("Population by Continent (2007)")
axes[2].set_xlabel("Population")
plt.tight_layout(); plt.show()
